# Factuality Metrics

Aggregates the per-row labels produced by the factuality pipeline
(`factuality_full.csv`) into paper-ready breakdowns.

Verifications covered (per row, set by the pipeline):
1. **Author** — `author_status` ∈ {`found`, `hallucinated`}
2. **Field** — `field_status` ∈ {`field_match`, `field_mismatch`, `field_unknown`, `not_applicable`}
3. **Seniority** — `seniority_status` ∈ {`seniority_match`, `seniority_mismatch`, `seniority_unknown`, `not_applicable`}
4. **Location** — `location_status` ∈ {`location_match`, `location_mismatch`, `location_unknown`, `not_applicable`}
5. **Ethnicity** — `perceived_ethnicity` ∈ {Asian, White, Black or African American, Hispanic or Latino, Unknown}

Breakdowns produced for every check: by `model`, `field`, `location`, `language`.

In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option('display.float_format', lambda x: f'{x:.1f}')
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

INPUT = Path('../../results/summary/factuality_full.csv')
DIMENSIONS = ['model', 'field', 'location', 'language']

In [ ]:
df = pd.read_csv(INPUT, low_memory=False)
print(f'Rows: {len(df):,}   Columns: {len(df.columns)}')
df[DIMENSIONS + ['author_status', 'field_status', 'seniority_status',
                 'location_status', 'perceived_ethnicity']].head()

## Helper

In [ ]:
def breakdown(data: pd.DataFrame, status_col: str, dim_col: str) -> pd.DataFrame:
    """
    Pivot rows=dim, cols=status_values, values=percent (one row sums to 100).
    Adds a final `n` column with the row total.
    """
    counts = data.groupby([dim_col, status_col]).size().unstack(fill_value=0)
    pct = counts.div(counts.sum(axis=1), axis=0) * 100
    pct = pct.round(1)
    pct['n'] = counts.sum(axis=1)
    pct = pct.sort_values('n', ascending=False)
    return pct


def overall(data: pd.DataFrame, status_col: str) -> pd.Series:
    counts = data[status_col].value_counts()
    pct = (counts / counts.sum() * 100).round(1)
    pct.name = f'{status_col} (% over n={len(data):,})'
    return pct

## 1. Author — `found` vs `hallucinated`

In [ ]:
overall(df, 'author_status')

In [ ]:
for dim in DIMENSIONS:
    print(f'\n=== author_status × {dim} ===')
    display(breakdown(df, 'author_status', dim))

## 2. Field — match vs mismatch vs unknown

Restricted to rows where the author was found (otherwise field is N/A).

In [ ]:
df_field = df[df['author_status'] == 'found'].copy()
print(f'Rows considered: {len(df_field):,}')
overall(df_field, 'field_status')

In [ ]:
for dim in DIMENSIONS:
    print(f'\n=== field_status × {dim} ===')
    display(breakdown(df_field, 'field_status', dim))

## 3. Seniority — match vs mismatch vs unknown

Restricted to rows where the author was found.

In [ ]:
df_sen = df[df['author_status'] == 'found'].copy()
overall(df_sen, 'seniority_status')

In [ ]:
for dim in DIMENSIONS:
    print(f'\n=== seniority_status × {dim} ===')
    display(breakdown(df_sen, 'seniority_status', dim))

## 4. Location — match vs mismatch vs unknown

Restricted to rows where the author was found.

In [ ]:
df_loc = df[df['author_status'] == 'found'].copy()
overall(df_loc, 'location_status')

In [ ]:
for dim in DIMENSIONS:
    print(f'\n=== location_status × {dim} ===')
    display(breakdown(df_loc, 'location_status', dim))

## 5. Ethnicity — distribution of `perceived_ethnicity`

All rows (ethnicity is name-based, independent of factuality).

In [ ]:
overall(df, 'perceived_ethnicity')

In [ ]:
for dim in DIMENSIONS:
    print(f'\n=== perceived_ethnicity × {dim} ===')
    display(breakdown(df, 'perceived_ethnicity', dim))

## 6. Combined — fully correct rows

A row is *fully correct* iff the author was found AND field/seniority/location all match.
Rows with any `unknown` are excluded from the denominator (they couldn't be evaluated).

In [ ]:
evaluable = df[
    (df['author_status']  == 'found')
    & df['field_status'].isin(['field_match', 'field_mismatch'])
    & df['seniority_status'].isin(['seniority_match', 'seniority_mismatch'])
    & df['location_status'].isin(['location_match', 'location_mismatch'])
].copy()

evaluable['fully_correct'] = (
    (evaluable['field_status']     == 'field_match')
    & (evaluable['seniority_status'] == 'seniority_match')
    & (evaluable['location_status']  == 'location_match')
)

print(f'Evaluable rows: {len(evaluable):,} / {len(df):,}')
print(f'Fully correct: {evaluable["fully_correct"].sum():,}  ({evaluable["fully_correct"].mean()*100:.1f}%)')

In [ ]:
for dim in DIMENSIONS:
    print(f'\n=== fully_correct × {dim} ===')
    g = evaluable.groupby(dim)['fully_correct']
    out = pd.DataFrame({
        'pct_fully_correct': (g.mean() * 100).round(1),
        'n':                 g.size(),
    }).sort_values('n', ascending=False)
    display(out)